# NYC Taxi tip prediction

原稿7.4のAgent Platform Workbench / Agent Platform Colab Enterprise用の最小例です。出力は保存しません。

In [ ]:
# Set this value for your Google Cloud environment.
BUCKET_NAME = "<YOUR_BUCKET_NAME>"
if BUCKET_NAME == "<YOUR_BUCKET_NAME>":
    raise ValueError("Set BUCKET_NAME before running this notebook.")

In [ ]:
import gcsfs
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [ ]:
fs = gcsfs.GCSFileSystem()
file_list = sorted(fs.glob(f"gs://{BUCKET_NAME}/data/nyc-taxi-tip-2022/taxi-*.csv"))
if not file_list:
    raise FileNotFoundError("No taxi-*.csv files were found under the configured bucket prefix.")
source_path = f"gs://{file_list[0]}"
print(f"Reading file: {source_path}")
df = pd.read_csv(source_path)
df.head()

In [ ]:
X = df.drop(columns=["tip_amount", "pickup_datetime"])
y = df["tip_amount"]
categorical_features = ["payment_type", "day_of_week", "hour_of_day"]
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [ ]:
model_lgbm = lgb.LGBMRegressor(random_state=42)
model_lgbm.fit(X_train, y_train)
y_pred = model_lgbm.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"LightGBM model RMSE: {rmse:.4f}")

In [ ]:
model_path = f"gs://{BUCKET_NAME}/models/nyc-taxi-tip/lgbm_model.joblib"
with fs.open(model_path, "wb") as handle:
    joblib.dump(model_lgbm, handle)
print(f"Saved model to {model_path}")